# ClipCap validation end-to-end on Google Colab

Notebook điều phối toàn bộ validation run cho năm Mapper:

1. Clone source code từ GitHub và mount Google Drive chứa artifact.
2. Chạy `clipcap_caption_generation.ipynb` để sinh caption.
3. Chạy `clipcap_evaluation.ipynb` để tính CIDEr, BLEU-4 và CLIPScore.
4. In bảng xếp hạng và đường dẫn artifact.

Notebook này cố định split `val`. Không dùng notebook này để tuning trên test.

## 1. Clone GitHub, mount Drive và đặt đường dẫn

Source code được clone vào `/content` để I/O nhanh. Drive chỉ lưu feature cache, checkpoint, caption inference và metric. Đặt `VALIDATION_CHUNK` từ 1 đến 9 để chạy một phần, hoặc `None` để chạy đủ 807 ảnh.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/HnhanBk415/zfs-clip-image-captioning.git'
BRANCH = 'refactor/huuthien/Model'
PROJECT_ROOT = Path('/content/zfs-clip-image-captioning')
if not PROJECT_ROOT.is_dir():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_ROOT)],
        check=True,
    )
else:
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'],
        check=True,
    )

DRIVE_ROOT = Path('/content/drive/MyDrive/clipcap_colab')
DATA_CACHE_DIR = DRIVE_ROOT / 'data_cache'
CHECKPOINT_ROOT = DRIVE_ROOT / 'experiments_fixed_epoch'
INFERENCE_OUTPUT_BASE = DRIVE_ROOT / 'evaluation_outputs'
METRICS_OUTPUT_BASE = DRIVE_ROOT / 'metrics'
INFERENCE_OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

VALIDATION_CHUNK: int | None = 1
FULL_VALIDATION_PATH = (
    PROJECT_ROOT / 'data' / 'flickr8k' / 'splits' / 'val.json'
)
if VALIDATION_CHUNK is None:
    VALIDATION_SCOPE = 'full'
    MANIFEST_PATH = FULL_VALIDATION_PATH
elif isinstance(VALIDATION_CHUNK, int) and 1 <= VALIDATION_CHUNK <= 9:
    VALIDATION_SCOPE = f'chunk_{VALIDATION_CHUNK:03d}'
    MANIFEST_PATH = (
        PROJECT_ROOT / 'data' / 'flickr8k' / 'splits' / 'val_chunks'
        / f'val_chunk_{VALIDATION_CHUNK:03d}.json'
    )
else:
    raise ValueError('VALIDATION_CHUNK phải là None hoặc số nguyên từ 1 đến 9')
REFERENCES_PATH = FULL_VALIDATION_PATH
FEATURE_CACHE_PATH = (
    DATA_CACHE_DIR / 'features' / 'clip_features.pt'
)
IMAGE_DIR = PROJECT_ROOT / 'data' / 'flickr8k' / 'raw' / 'Images'

required_paths = {
    'project': PROJECT_ROOT,
    'checkpoint root': CHECKPOINT_ROOT,
    'selected validation manifest': MANIFEST_PATH,
    'full validation references': REFERENCES_PATH,
}
for label, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(f'Không tìm thấy {label}: {path}')
if not FEATURE_CACHE_PATH.is_file() and not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        'Cần feature cache hoặc thư mục ảnh để chạy inference validation.'
    )

os.chdir(PROJECT_ROOT)
print(f'Working directory: {Path.cwd()}')
print(f'Git branch: {BRANCH}')
print(f'Drive root: {DRIVE_ROOT}')
print(f'Checkpoint root: {CHECKPOINT_ROOT}')
print(f'Feature cache: {FEATURE_CACHE_PATH}')
print(f'Validation scope: {VALIDATION_SCOPE}')
print(f'Validation manifest: {MANIFEST_PATH}')

## 2. Cài và kiểm tra dependencies

Lần đầu chạy có thể mất vài phút. Các lần sau có thể đặt `INSTALL_DEPENDENCIES = False` nếu runtime vẫn còn package. CIDEr/BLEU-4 chuẩn COCO cần Java.

In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
        check=True,
    )
    if importlib.util.find_spec('pycocoevalcap') is None:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', 'pycocoevalcap'],
            check=True,
        )

if shutil.which('java') is None:
    raise RuntimeError('Java không có trong PATH; chưa thể tính CIDEr/BLEU-4')
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['java', '-version'], check=True)
print('Colab environment is ready')

## 3. Cấu hình validation experiment

Đổi `RUN_TAG` mỗi khi thay đổi cấu hình inference. Để tuning nhanh, giữ cố định cùng một `VALIDATION_CHUNK` cho mọi cấu hình. Sau khi chốt tham số, đặt `VALIDATION_CHUNK = None` để chạy đủ 807 ảnh.

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.config.clipcap_config import (
    CLIPCAP_DEFAULT_INFERENCE_CONFIG,
    CLIPCAP_TRAIN_SEED,
    CLIPCAP_TRAIN_SUBSETS,
)

RUN_TAG = 'baseline_v1'
SEED = CLIPCAP_TRAIN_SEED
SUBSETS = CLIPCAP_TRAIN_SUBSETS
MAX_NEW_TOKENS = CLIPCAP_DEFAULT_INFERENCE_CONFIG.max_new_tokens
NUM_BEAMS = CLIPCAP_DEFAULT_INFERENCE_CONFIG.num_beams
NUM_RETURN_SEQUENCES = (
    CLIPCAP_DEFAULT_INFERENCE_CONFIG.num_return_sequences
)
LENGTH_PENALTY = CLIPCAP_DEFAULT_INFERENCE_CONFIG.length_penalty
EARLY_STOPPING = CLIPCAP_DEFAULT_INFERENCE_CONFIG.early_stopping
IMAGE_BATCH_SIZE = CLIPCAP_DEFAULT_INFERENCE_CONFIG.image_batch_size
METRIC_BATCH_SIZE = 32

EFFECTIVE_RUN_TAG = (
    RUN_TAG
    if VALIDATION_CHUNK is None
    else f'{RUN_TAG}_val_chunk_{VALIDATION_CHUNK:03d}'
)

environment = {
    'ZFS_CLIP_PROJECT_ROOT': str(PROJECT_ROOT),
    'ZFS_CLIP_SPLIT_NAME': 'val',
    'ZFS_CLIP_ALLOW_TEST': '0',
    'ZFS_CLIP_RUN_TAG': EFFECTIVE_RUN_TAG,
    'ZFS_CLIP_RUN_INFERENCE': '1',
    'ZFS_CLIP_SEED': str(SEED),
    'ZFS_CLIP_SUBSETS': ','.join(SUBSETS),
    'ZFS_CLIP_INFERENCE_MANIFEST_PATH': str(MANIFEST_PATH),
    'ZFS_CLIP_REFERENCES_PATH': str(REFERENCES_PATH),
    'ZFS_CLIP_CHECKPOINT_ROOT': str(CHECKPOINT_ROOT),
    'ZFS_CLIP_INFERENCE_OUTPUT_BASE': str(INFERENCE_OUTPUT_BASE),
    'ZFS_CLIP_METRICS_OUTPUT_BASE': str(METRICS_OUTPUT_BASE),
    'ZFS_CLIP_FEATURE_CACHE': str(FEATURE_CACHE_PATH),
    'ZFS_CLIP_DEVICE': 'cuda',
    'ZFS_CLIP_IMAGE_BATCH_SIZE': str(IMAGE_BATCH_SIZE),
    'ZFS_CLIP_METRIC_BATCH_SIZE': str(METRIC_BATCH_SIZE),
    'ZFS_CLIP_MAX_NEW_TOKENS': str(MAX_NEW_TOKENS),
    'ZFS_CLIP_NUM_BEAMS': str(NUM_BEAMS),
    'ZFS_CLIP_NUM_RETURN_SEQUENCES': str(NUM_RETURN_SEQUENCES),
    'ZFS_CLIP_LENGTH_PENALTY': str(LENGTH_PENALTY),
    'ZFS_CLIP_EARLY_STOPPING': '1' if EARLY_STOPPING else '0',
    'ZFS_CLIP_REFERENCES_PER_IMAGE': '5',
}
if IMAGE_DIR.is_dir():
    environment['ZFS_CLIP_IMAGE_DIR'] = str(IMAGE_DIR)
for name, value in environment.items():
    os.environ[name] = value

print(json.dumps({
    'split': 'val',
    'base_run_tag': RUN_TAG,
    'effective_run_tag': EFFECTIVE_RUN_TAG,
    'validation_chunk': VALIDATION_CHUNK,
    'subsets': list(SUBSETS),
    'max_new_tokens': MAX_NEW_TOKENS,
    'num_beams': NUM_BEAMS,
    'num_return_sequences': NUM_RETURN_SEQUENCES,
    'length_penalty': LENGTH_PENALTY,
    'early_stopping': EARLY_STOPPING,
}, indent=2))

## 4. Chạy inference rồi tính metric

Inference có resume. Nếu Colab bị ngắt, chạy lại với cùng `RUN_TAG` và `VALIDATION_CHUNK`; các ảnh hoàn thành sẽ được bỏ qua. Metric chỉ bắt đầu sau khi cả năm prediction file của chunk đã đầy đủ.

In [ ]:
inference_notebook = (
    PROJECT_ROOT / 'notebook' / 'clipcap' / 'inference' / 'clipcap_caption_generation.ipynb'
)
evaluation_notebook = (
    PROJECT_ROOT / 'notebook' / 'evaluation' / 'clipcap_evaluation.ipynb'
)
for path in (inference_notebook, evaluation_notebook):
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy notebook: {path}')

ipython = get_ipython()
print('Starting ClipCap validation inference...')
ipython.run_line_magic('run', str(inference_notebook))
print('Starting ClipCap validation metrics...')
ipython.run_line_magic('run', str(evaluation_notebook))

## 5. Đọc bảng xếp hạng cuối

CIDEr dùng để chọn cấu hình chính; BLEU-4 dùng khi CIDEr gần nhau. CLIPScore không phải tiêu chí tuning chính.

In [ ]:
import pandas as pd
from IPython.display import display

summary_path = (
    METRICS_OUTPUT_BASE / 'val' / EFFECTIVE_RUN_TAG
    / f'seed_{SEED}' / 'summary.json'
)
per_image_path = summary_path.parent / 'per_image_scores.csv'
if not summary_path.is_file() or not per_image_path.is_file():
    raise FileNotFoundError('Metric artifacts chưa được tạo đầy đủ')
with summary_path.open('r', encoding='utf-8') as file:
    summary = json.load(file)
ranking = pd.DataFrame(summary['results']).sort_values('rank')
display(ranking[
    ['rank', 'experiment', 'CIDEr', 'BLEU-4', 'CLIPScore']
])
print(f'Summary: {summary_path}')
print(f'Per-image scores: {per_image_path}')
print('Validation end-to-end completed')

## 6. Sau validation

Metric trên 100 ảnh chỉ dùng để tuning nhanh và mọi cấu hình phải dùng cùng một chunk. Không lấy trung bình metric của các chunk làm metric toàn validation. Sau khi chốt cấu hình, đặt `VALIDATION_CHUNK = None` để chạy đủ 807 ảnh; sau đó mới chạy final test và không tuning theo kết quả test.